In [ ]:
import sys
sys.path.append('..')

import torch
import json
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from src.utils import get_device, get_quantum_device, set_seed
from src.dataset import get_dataloaders
from src.models import HybridModel
from src.train import train

set_seed(42)
device = get_quantum_device()
print(f"Device: {device}")

In [ ]:
model = HybridModel()
print(model)

dummy = torch.randn(2, 3, 64, 64)
out = model(dummy)
print(f"Output shape: {out.shape}")

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(batch_size=32, augment=True)
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
from torch.utils.data import Subset, DataLoader

small_train = Subset(train_loader.dataset, range(64))
small_val   = Subset(val_loader.dataset, range(32))

small_train_loader = DataLoader(small_train, batch_size=8, shuffle=True)
small_val_loader   = DataLoader(small_val, batch_size=8, shuffle=False)

In [ ]:
import time

start = time.time()
history = train(model, small_train_loader, small_val_loader,
                epochs=1, lr=0.0005, device=device, save_name='hybrid_vqc_smoketest')
print(f"\nTime for 1 epoch on 64 samples: {time.time() - start:.1f}s")

In [ ]:
import numpy as np
from torchvision import datasets
from torch.utils.data import Subset, DataLoader, random_split
from src.dataset import get_transforms, DATA_DIR

# Load full EuroSAT dataset with training transforms
full_dataset = datasets.EuroSAT(root=DATA_DIR, download=True, transform=get_transforms(augment=True))

# Balanced sampling — 200 images per class
targets = np.array(full_dataset.targets)
samples_per_class = 200

balanced_indices = []
for class_idx in range(10):
    class_indices = np.where(targets == class_idx)[0]
    chosen = np.random.choice(class_indices, samples_per_class, replace=False)
    balanced_indices.extend(chosen)

np.random.shuffle(balanced_indices)
balanced_subset = Subset(full_dataset, balanced_indices)

# 80/20 train/val split
n = len(balanced_subset)
train_n = int(0.8 * n)
val_n = n - train_n
train_subset, val_subset = random_split(balanced_subset, [train_n, val_n])

vqc_train_loader = DataLoader(train_subset, batch_size=8, shuffle=True)
vqc_val_loader   = DataLoader(val_subset, batch_size=8, shuffle=False)

print(f"Total: {n} | Train: {len(train_subset)} | Val: {len(val_subset)}")

In [ ]:
import time

model = HybridModel()
start = time.time()
history = train(model, vqc_train_loader, vqc_val_loader,
                epochs=5, lr=0.0005, device=device, save_name='hybrid_vqc_quick')
print(f"\nTotal time: {(time.time()-start)/60:.1f} min")

In [ ]:
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for images, labels in vqc_val_loader:
        preds = model(images).argmax(dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

import collections
print("Predicted class distribution:", collections.Counter(all_preds))
print("True class distribution:    ", collections.Counter(all_labels))

In [ ]:
model.eval()
with torch.no_grad():
    # Grab a small batch of real images
    images, labels = next(iter(vqc_val_loader))

    # Step through the pipeline manually
    x = model.features(images)
    x = x.view(x.size(0), -1)
    x = torch.relu(model.fc(x))          # before dropout, 256-dim features

    pre_out = model.vqc.pre(x)           # input to quantum circuit, 4-dim
    print("Pre-quantum (input to circuit) — first 5 samples:")
    print(pre_out[:5])

    q_out = torch.stack([model.vqc.qlayer(pre_out[i]) for i in range(pre_out.shape[0])])
    print("\nQuantum circuit output — first 5 samples:")
    print(q_out[:5])

In [ ]:
model.eval()
with torch.no_grad():
    images, labels = next(iter(vqc_val_loader))

    x = model.features(images)
    x = x.view(x.size(0), -1)
    x = torch.relu(model.fc(x))

    pre_out = model.vqc.pre(x)
    print("Pre-quantum (input to circuit) — first 5 samples:")
    print(pre_out[:5])

    q_out = torch.stack([model.vqc.qlayer(pre_out[i]) for i in range(pre_out.shape[0])])
    print("\nQuantum circuit output — first 5 samples:")
    print(q_out[:5])

In [ ]:
model.eval()
with torch.no_grad():
    images, labels = next(iter(vqc_val_loader))

    x = model.features(images)
    x = x.view(x.size(0), -1)
    x = torch.relu(model.fc(x))

    pre_out = model.vqc.pre(x)
    print("Pre-quantum (input to circuit) — first 5 samples:")
    print(pre_out[:5])

    q_out = torch.stack([model.vqc.qlayer(pre_out[i]) for i in range(pre_out.shape[0])])
    print("\nQuantum circuit output — first 5 samples:")
    print(q_out[:5])

In [ ]:
import time

model = HybridModel()
start = time.time()
history = train(model, vqc_train_loader, vqc_val_loader,
                epochs=5, lr=0.0005, device=device, save_name='hybrid_vqc_quick')
print(f"\nTotal time: {(time.time()-start)/60:.1f} min")

In [ ]:
import numpy as np
from torchvision import datasets
from torch.utils.data import Subset, DataLoader, random_split
from src.dataset import get_transforms, DATA_DIR

full_dataset = datasets.EuroSAT(root=DATA_DIR, download=True, transform=get_transforms(augment=True))

targets = np.array(full_dataset.targets)
samples_per_class = 600

balanced_indices = []
for class_idx in range(10):
    class_indices = np.where(targets == class_idx)[0]
    chosen = np.random.choice(class_indices, samples_per_class, replace=False)
    balanced_indices.extend(chosen)

np.random.shuffle(balanced_indices)
balanced_subset_full = Subset(full_dataset, balanced_indices)

n = len(balanced_subse

In [ ]:
import numpy as np
from torchvision import datasets
from torch.utils.data import Subset, DataLoader, random_split
from src.dataset import get_transforms, DATA_DIR

full_dataset = datasets.EuroSAT(root=DATA_DIR, download=True, transform=get_transforms(augment=True))

targets = np.array(full_dataset.targets)
samples_per_class = 600

balanced_indices = []
for class_idx in range(10):
    class_indices = np.where(targets == class_idx)[0]
    chosen = np.random.choice(class_indices, samples_per_class, replace=False)
    balanced_indices.extend(chosen)

np.random.shuffle(balanced_indices)
balanced_subset_full = Subset(full_dataset, balanced_indices)

n = len(balanced_subset_full)
train_n = int(0.8 * n)
val_n = n - train_n
train_subset_full, val_subset_full = random_split(balanced_subset_full, [train_n, val_n])

vqc_train_loader_full = DataLoader(train_subset_full, batch_size=8, shuffle=True)
vqc_val_loader_full   = DataLoader(val_subset_full, batch_size=8, shuffle=False)

print(f"Total: {n} | Train: {len(train_subset_full)} | Val: {len(val_subset_full)}")

In [ ]:
import time

model = HybridModel()
start = time.time()
history = train(model, vqc_train_loader_full, vqc_val_loader_full,
                epochs=15, lr=0.0005, device=device, save_name='hybrid_vqc_full')
print(f"\nTotal time: {(time.time()-start)/60:.1f} min")

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load best checkpoint
model = HybridModel()
model.load_state_dict(torch.load('../checkpoints/hybrid_vqc_full_best.pth', map_location=device))
model.to(device)
model.eval()

classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway',
           'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in vqc_val_loader_full:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

with open('../results/vqc_results.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes,
            yticklabels=classes, cmap='Purples')
plt.title('HybridModel (VQC) Confusion Matrix — 6000-image subset')
plt.tight_layout()
plt.savefig('../results/vqc_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import time

model = HybridModel()
start = time.time()
history = train(model, train_loader, val_loader,
                epochs=15, lr=0.0005, device=device, save_name='hybrid_vqc_fulldataset')
print(f"\nTotal time: {(time.time()-start)/3600:.2f} hours")

In [ ]:
model = HybridModel()
model.load_state_dict(torch.load('../checkpoints/hybrid_vqc_fulldataset_best.pth', map_location=device))
model.to(device)
model.eval()

classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway',
           'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

with open('../results/vqc_fulldataset_results.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
model = HybridModel()
model.load_state_dict(torch.load('../checkpoints/hybrid_vqc_fulldataset_best.pth', map_location=device))
model.to(device)
model.eval()

classes = ['AnnualCrop','Forest','HerbaceousVegetation','Highway',
           'Industrial','Pasture','PermanentCrop','Residential','River','SeaLake']

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

report = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

with open('../results/vqc_fulldataset_results.json', 'w') as f:
    json.dump(report, f, indent=2)

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes,
            yticklabels=classes, cmap='Purples')
plt.title('HybridModel (VQC) Confusion Matrix — Full Dataset, Test Set')
plt.tight_layout()
plt.savefig('../results/vqc_fulldataset_confusion_matrix.png', dpi=150)
plt.show()